In [26]:
import numpy as np
import pandas as pd


In [27]:
def load_data(split_n):
    cols = ['user_id','movie_id','rating','timestamp']

    var = f"u{split_n}"
    
    train_path = f"ml-100k/{var}.base"
    test_path = f"ml-100k/{var}.test"

    train = pd.read_csv(train_path, sep='\t', names = cols)
    test = pd.read_csv(test_path, sep='\t', names = cols)

    print(f"Split {var} - Training: {len(train)} ratings, Test: {len(test)} ratings")

    return train, test


def create_matrix(train):

    matrix = train.pivot_table(
        index='user_id', 
        columns='movie_id', 
        values='rating'
    ).fillna(0)

    return matrix



def stats(train):
    overall_mean = train['rating'].mean()
    user_means = train.groupby('user_id')['rating'].mean()
    item_means = train.groupby('movie_id')['rating'].mean()

    user_b = user_means- overall_mean
    item_b = item_means - overall_mean

    return overall_mean, user_means, item_means, user_b, item_b


In [28]:
def pearson_similarity(matrix, axis =1, min_support=40):

    matrix_with_nan = np.where(matrix==0, np.nan, matrix)

    means = np.nanmean(matrix_with_nan, axis=axis, keepdims = True)
    centered = matrix_with_nan - means
    centered = np.nan_to_num(centered, 0)

    norms = np.sqrt(np.sum(centered**2, axis = axis, keepdims = True))
    norms = np.where(norms==0,1,norms)
    normalised = centered/norms
    sim = normalised @ normalised.T

    if(axis==1):
        support = (matrix>0).astype(int) @ (matrix>0).astype(int).T
    else:
        support = (matrix>0).astype(int).T @ (matrix>0).astype(int)

    significance = np.minimum(support/min_support, 1.0)
    sim = sim* significance 

    return sim


def all_sims(matrix, min_support =40):

    # print("USER SIM MATRIX->")
    user_sim = pearson_similarity(matrix.values, axis =1, min_support = min_support)

    # print("ITEM SIM MATRIX->")
    item_sim = pearson_similarity(matrix.T.values, axis =1, min_support = min_support)

    return user_sim, item_sim



In [29]:
## user based collaborative filtering 
def predict_user(user_id, item_id, matrix, user_sim, user_means, overall_mean, user_b, k_users =40):

    # basically returning avg rating + user bias if no users have rated that item or no users have positively rated that item
    # or returning avg rating +  summation_over_k_neighbours(sim * (rating- mean_of_neighbourhood))/(total_sim_sum)
    users = matrix.index.values
    movie = matrix.columns.values

    if user_id not in users or item_id not in movie:
        return overall_mean
        
    user_idx = np.where(users == user_id)[0][0]
    item_idx = np.where(movie == item_id)[0][0]

    item_ratings = matrix.iloc[:, item_idx].values
    rated_users  = np.where(item_ratings>0)[0]

    if(len(rated_users)==0):
        return overall_mean + user_b.get(user_id,0)

    user_similarity = user_sim[user_idx, rated_users]

    if(len(user_similarity)> k_users):
        best_k = np.argsort(user_similarity)[-k_users:]
        user_similarity = user_similarity[best_k]
        rated_users = rated_users[best_k]

    pos_sims = user_similarity > 0
    if not pos_sims.any():
        return overall_mean + user_b.get(user_id, 0)

    user_similarity = user_similarity[pos_sims]
    rated_users = rated_users[pos_sims]

    neighbor_ratings = item_ratings[rated_users]
    neighbor_means = user_means.loc[users[rated_users]].values

    weighted_sum = np.sum(user_similarity * (neighbor_ratings - neighbor_means))
    sim_sum = np.sum(user_similarity)
    
    if sim_sum == 0:
        return overall_mean + user_b.get(user_id, 0)
    
    prediction = user_means.get(user_id, overall_mean) + (weighted_sum / sim_sum)
    return prediction




In [30]:
def predict_item(user_id, item_id, matrix, item_sim, item_means, overall_mean, item_b, k_items =40):
    users = matrix.index.values
    movie = matrix.columns.values

    if user_id not in users or item_id not in movie:
        return overall_mean
        
    user_idx = np.where(users == user_id)[0][0]
    item_idx = np.where(movie == item_id)[0][0]

    user_ratings = matrix.iloc[user_idx, :].values
    rated_items = np.where(user_ratings>0)[0]

    if(len(rated_items) ==0):
        return overall_mean + item_b.get(item_id,0)

    item_similarity = item_sim[item_idx,rated_items]
    if(len(item_similarity)>k_items):
        best_k = np.argsort(item_similarity)[-k_items:]
        item_similarity = item_similarity[best_k]
        rated_items = rated_items[best_k]

    pos_sims = item_similarity>0
    if not pos_sims.any():
        return overall_mean + item_b.get(item_id,0)

    item_similarity = item_similarity[pos_sims]
    rated_items = rated_items[pos_sims]

    neighbor_ratings = user_ratings[rated_items]
    neighbor_means = item_means.loc[movie[rated_items]].values

    wt_sum = np.sum(item_similarity*(neighbor_ratings-neighbor_means))
    sim_sum = np.sum(item_similarity)

    if(sim_sum==0):
        return overall_mean + item_b.get(item_id,0)

    pred = item_means.get(item_id, overall_mean) + (wt_sum/sim_sum)
    return pred



    

In [31]:
def predict_hybrid(user_id, item_id, user_item_matrix, user_similarity, 
                   item_similarity, user_means, item_means, global_mean, 
                   user_bias, item_bias, k_users=40, k_items=40, alpha=0.1):
    """
    Hybrid prediction combining user-based and item-based approaches
    Uses adaptive weighting based on prediction confidence
    """
    user_pred = predict_user(
        user_id, item_id, user_item_matrix, user_similarity, 
        user_means, global_mean, user_bias, k_users
    )
    
    item_pred = predict_item(
        user_id, item_id, user_item_matrix, item_similarity, 
        item_means, global_mean, item_bias, k_items
    )
    
    users = user_item_matrix.index.values
    items = user_item_matrix.columns.values
    
    user_in_train = user_id in users
    item_in_train = item_id in items
    
    if user_in_train and item_in_train:
        user_idx = np.where(users == user_id)[0][0]
        item_idx = np.where(items == item_id)[0][0]
        
        user_rating_count = np.sum(user_item_matrix.iloc[user_idx, :] > 0)
        item_rating_count = np.sum(user_item_matrix.iloc[:, item_idx] > 0)
        
        total_counts = user_rating_count + item_rating_count
        if total_counts > 0:
            adaptive_alpha = user_rating_count / total_counts
        else:
            adaptive_alpha = alpha
    else:
        adaptive_alpha = alpha
    
    hybrid_pred = adaptive_alpha * user_pred + (1 - adaptive_alpha) * item_pred
    
    return np.clip(hybrid_pred, 1, 5)

In [32]:
def evaluate_predictions(test_df, user_item_matrix, user_similarity, 
                        item_similarity, user_means, item_means, global_mean,
                        user_bias, item_bias, k_users=40, k_items=40, alpha=0.1):
    """Generate predictions and calculate NMAE"""
    predictions = []
    actuals = []
    
    print("Generating predictions...")
    for idx, row in test_df.iterrows():
        pred = predict_hybrid(
            row['user_id'], row['movie_id'], user_item_matrix, 
            user_similarity, item_similarity, user_means, item_means,
            global_mean, user_bias, item_bias, k_users, k_items, alpha
        )
        predictions.append(pred)
        actuals.append(row['rating'])
        
        if (idx + 1) % 5000 == 0:
            print(f"  Processed {idx + 1}/{len(test_df)} predictions")
    
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    
    # Calculate NMAE
    mae = np.mean(np.abs(predictions - actuals))
    nmae = mae / 4.0  # Normalize by rating range (5-1=4)
    
    return nmae, predictions, actuals

In [33]:
def run_for_1_split(split_n, k_users =40, k_items=40, min_support =40,alpha=0.1):

    print(f"RUNNING HYBRID CF ON SPLIT u{split_n}")
    print(f"Parameters: k_users={k_users}, k_items={k_items}, "f"min_support={min_support}, alpha={alpha}")

    train, test = load_data(split_n)
    matrix = create_matrix(train)
    overall_mean, user_means, item_means, user_b, item_b = stats(train)
    user_sim, item_sim = all_sims(matrix,min_support)
    nmae, pred, true_value = evaluate_predictions(test,matrix,user_sim, item_sim,user_means, item_means,
                                                  overall_mean, user_b, item_b, k_users, k_items, alpha)


    print("\n", f"SPLIT u{split_n} RESULT: NMAE = {nmae:.6f}","\n")
    return nmae;

    

In [34]:
## MAIN SCRIPT TO RUN ON ALL SPLITS AND DISPLAY NMAE

print("Running Hybird CF on all splits- u1, u2, u3, u4 and u5")

k_users = 40
k_items = 40
min_support = 40
alpha = 0.5

print("\nThe parameters are:")
print(f"k_users for user based CF: {k_users}")
print(f"k_items for item based CF: {k_items}")
print(f"N for significance weighting: {min_support}")
print(f"alpha for Hybrid CF:{alpha}")
print("\n")
NMAE =[]

for i in range(1,6):
    nmae = run_for_1_split(i,k_users=k_users, k_items = k_items, min_support = min_support,alpha =alpha)
    NMAE.append(nmae)


print("Result for all splits are: ")
for i in range(5):
    print(f"NMAE for split {i+1} is:{NMAE[i]}","\n")

print(f"{'Average':<10} {np.mean(NMAE):.6f}")
print(f"{'Std Dev':<10} {np.std(NMAE):.6f}")

Running Hybird CF on all splits- u1, u2, u3, u4 and u5

The parameters are:
k_users for user based CF: 40
k_items for item based CF: 40
N for significance weighting: 40
alpha for Hybrid CF:0.5


RUNNING HYBRID CF ON SPLIT u1
Parameters: k_users=40, k_items=40, min_support=40, alpha=0.5
Split u1 - Training: 80000 ratings, Test: 20000 ratings
Generating predictions...
  Processed 5000/20000 predictions
  Processed 10000/20000 predictions
  Processed 15000/20000 predictions
  Processed 20000/20000 predictions

 SPLIT u1 RESULT: NMAE = 0.181445 

RUNNING HYBRID CF ON SPLIT u2
Parameters: k_users=40, k_items=40, min_support=40, alpha=0.5
Split u2 - Training: 80000 ratings, Test: 20000 ratings
Generating predictions...
  Processed 5000/20000 predictions
  Processed 10000/20000 predictions
  Processed 15000/20000 predictions
  Processed 20000/20000 predictions

 SPLIT u2 RESULT: NMAE = 0.179384 

RUNNING HYBRID CF ON SPLIT u3
Parameters: k_users=40, k_items=40, min_support=40, alpha=0.5
Split